# Options ML Scoring - Example Inference

Standalone notebook that loads the V7 XGBoost models, scores a sample option candidate,
shows the composite ML score, and provides a SHAP explanation.

No FastAPI server needed - just run the cells.

In [ ]:
import json
import numpy as np
import xgboost as xgb

# Load V7 models
models = {}
for name in ['hit50', 'maxprofit', 'days50', 'ev', 'outcome']:
    m = xgb.XGBClassifier() if name in ('hit50', 'outcome') else xgb.XGBRegressor()
    m.load_model(f'../models/v7/{name}_model.json')
    models[name] = m

with open('../models/v7/model_meta.json') as f:
    meta = json.load(f)

print(f"Loaded {len(models)} models")
print(f"Features: {len(meta['feature_columns'])}")
print(f"Tickers: {len(meta['tickers'])}")

## Sample Option Candidate

In [ ]:
# NVDA $130 PUT, 30 DTE, selling at $3.50
candidate = {
    'dte': 30, 'delta': -0.20, 'gamma': 0.015, 'theta': -0.08, 'vega': 0.25,
    'iv': 0.45, 'ivr': 65, 'ivp': 72, 'moneyness': 0.93,
    'annualized_return': 0.035, 'credit_received': 350, 'max_loss': 13000,
    'rsi14': 55, 'macd_signal': 1, 'bb_position': 0.6, 'resistance_score': 0.7,
    'days_to_earnings': float('nan'), 'earnings_in_window': 0, 'post_earnings': 0,
    'credit_pct': 0.027, 'iv_vs_ticker_avg': 1.1, 'return_5d': 0.02, 'return_10d': -0.01,
    'return_20d': 0.05, 'rv_20d': 0.35, 'rv_iv_ratio': 0.78, 'iv_skew_proxy': 0.03,
    'dte_bucket': 1, 'theta_vega_ratio': 0.32, 'delta_moneyness': 0.186,
}

X = np.array([[candidate[f] for f in meta['feature_columns']]])
print("Feature vector shape:", X.shape)
print("\nCandidate: NVDA $130 PUT, 30 DTE, $3.50 credit")

## Model Predictions

In [ ]:
# Model 1: P(hit 50% profit)
hit50_prob = models['hit50'].predict_proba(X)[0][1]
print(f"P(hit 50% profit): {hit50_prob:.1%}")

# Model 2: Max profit at expiry
max_profit = models['maxprofit'].predict(X)[0]
print(f"Predicted max profit: {max_profit:.1%}")

# Model 3: Days to 50% (only meaningful if hit)
days_50 = max(1, models['days50'].predict(X)[0])
print(f"Predicted days to 50%: {days_50:.1f} days")

# Model 4: Expected value ($)
ev = models['ev'].predict(X)[0]
ev_clamped = max(-(13000 - 350), min(350, ev))  # physical bounds
print(f"Predicted EV: ${ev_clamped:.0f}")

# Model 5: Outcome category
outcome_classes = ['breakeven', 'full_win', 'loss', 'partial_win']
outcome_probs = models['outcome'].predict_proba(X)[0]
pred_outcome = outcome_classes[np.argmax(outcome_probs)]
print(f"\nOutcome: {pred_outcome}")
for cls, prob in zip(outcome_classes, outcome_probs):
    bar = '\u2588' * int(prob * 40)
    print(f"  {cls:15s} {prob:6.1%} {bar}")

# Composite score
score_hit = hit50_prob * 100
score_profit = max(0, min(100, (max_profit + 1) * 50))
score_speed = max(0, min(100, (1 - days_50 / 45) * 100))
ml_score = 0.50 * score_hit + 0.30 * score_profit + 0.20 * score_speed
print(f"\nComposite ML Score: {ml_score:.1f}/100")
print(f"Passes threshold (>=0.85): {'Yes' if hit50_prob >= 0.85 else 'No'}")

## SHAP Explanation

In [ ]:
try:
    import shap
    explainer = shap.TreeExplainer(models['hit50'])
    shap_values = explainer.shap_values(X)

    print("Top factors for this prediction:\n")
    feature_shap = list(zip(meta['feature_columns'], shap_values[0]))
    feature_shap.sort(key=lambda x: abs(x[1]), reverse=True)

    for feat, sv in feature_shap[:10]:
        direction = '+' if sv > 0 else '-'
        bar = '\u2588' * int(abs(sv) * 20)
        val = candidate[feat]
        print(f"  {feat:25s} = {val:>8}  {direction} {sv:+.3f}  {bar}")
except ImportError:
    print("Install shap for explanations: pip install shap")

## Compare V6 vs V7

In [ ]:
# V6 uses only 19 features
with open('../models/v6/model_meta.json') as f:
    meta_v6 = json.load(f)

v6_hit50 = xgb.XGBClassifier()
v6_hit50.load_model('../models/v6/hit50_model.json')

# Use only V6's 19 features
X_v6 = np.array([[candidate[f] for f in meta_v6['feature_columns']]])
v6_prob = v6_hit50.predict_proba(X_v6)[0][1]

print(f"V6 prediction (19 features, 6 tickers):  {v6_prob:.1%}")
print(f"V7 prediction (30 features, 28 tickers): {hit50_prob:.1%}")
print(f"\nV6 training: {meta_v6['training_rows']:,} rows, {len(meta_v6['tickers'])} tickers")
print(f"V7 training: {meta['training_rows']:,} rows, {len(meta['tickers'])} tickers")